# README
## Purpose
Clean and normalize GSIB headlines for modeling and evaluation.
## Inputs
- `Data/GSIB_clean.csv`
## Outputs
- Preprocessed GSIB model-ready table (saved in notebook workflow as `Data/GSIB_model_input.csv`).
## Notes
Includes language filtering and entity masking to reduce ticker-dominated topics.

Imports

In [9]:
import re
import unicodedata
import numpy as np
import pandas as pd
from langdetect import detect_langs, LangDetectException

## Loading Data

In [10]:
data = pd.read_csv('Data/GSIB_clean.csv')
print('Raw shape:', data.shape)
display(data.head(5))

Raw shape: (126941, 4)


,stock,headline,date,date_only
0,JPM,JPMorgan Chase and Syracuse University Renew P...,2016-02-29 15:07:00+00:00,2016-02-29
1,JPM,"Cardtronics buys 2,586 off-premise ATMs from C...",2016-04-28 11:30:00+00:00,2016-04-28
2,JPM,Grameen Foundation and ideas42 Launch Partners...,2016-04-21 14:06:00+00:00,2016-04-21
3,JPM,Veteran Launch Addresses Unmet Financing Needs...,2016-03-22 14:00:00+00:00,2016-03-22
4,JPM,The Madison Square Garden Company Announces th...,2016-03-10 18:17:00+00:00,2016-03-10


## Preprocessing configuration and cleaning

In [11]:
# Choose modeling text strategy: 'keep', 'mask', or 'remove'
ENTITY_MODE = 'mask'

# Language detection: keep only English headlines
LANGUAGE_FILTER = 'en'  # ISO 639-1 language code for English
MIN_CONFIDENCE_LANG = 0.5  # Minimum confidence threshold for English detection

# Full ticker universe for masking
bank_tickers = [
    'JPM', 'BAC', 'C', 'HSBC', 'IDCBY', 'GS', 'BNPQY', 'UBS', 'ACGBY', 'BACHY',
    'CICHY', 'BCS', 'MUFG', 'MS', 'WFC', 'BK', 'STT', 'DB', 'ING', 'SAN', 'RY',
    'TD', 'SCBFF', 'SCGLY', 'MFG', 'SMFG', 'BCMXY', 'GCRLY', 'BPCE'
 ]
BANK_TICKERS = sorted({ticker.upper().strip() for ticker in bank_tickers})

# Optional: map ticker symbols to common aliases to reduce entity-dominated topics
STOCK_ALIAS_MAP = {
    'JPM': [r'jpmorgan', r'j\.p\.\s*morgan'],
    'BAC': [r'bank\s+of\s+america', r'bofa'],
    'WFC': [r'wells\s+fargo'],
    'C': [r'citigroup', r'\bciti\b'],
    'GS': [r'goldman\s+sachs'],
    'MS': [r'morgan\s+stanley'],
    'HSBC': [r'\bhsbc\b'],
    'SAN': [r'santander', r'banco\s+santander'],
    'BCS': [r'barclays'],
    'DB': [r'deutsche\s+bank'],
    'UBS': [r'\bubs\b'],
    'RY': [r'royal\s+bank\s+of\s+canada', r'\brbc\b'],
    'TD': [r'toronto\-dominion', r'toronto\s+dominion'],
    'SCBFF': [r'standard\s+chartered', r'\bstanchart\b'],
    'SCGLY': [r'societe\s+generale', r'société\s+générale', r'\bsocgen\b'],
    'MFG': [r'mizuho\s+financial\s+group', r'\bmizuho\b'],
    'SMFG': [r'sumitomo\s+mitsui', r'\bsmbc\b'],
    'MUFG': [r'mitsubishi\s+ufj', r'\bufj\b'],
    'BK': [r'bank\s+of\s+new\s+york\s+mellon', r'\bbny\b'],
    'STT': [r'state\s+street'],
    'IDCBY': [r'industrial\s+and\s+commercial\s+bank\s+of\s+china', r'\bicbc\b'],
    'ACGBY': [r'agricultural\s+bank\s+of\s+china'],
    'CICHY': [r'china\s+construction\s+bank'],
    'BACHY': [r'bank\s+of\s+china'],
    'BCMXY': [r'bank\s+of\s+communications'],
    'ING': [r'\bing\b\s+groep'],
    'BNPQY': [r'bnp\s+paribas'],
    'GCRLY': [r'credit\s+agricole', r'crédit\s+agricole'],
    'BPCE': [r'\bbpce\b']
}

print('ENTITY_MODE:', ENTITY_MODE)
print('LANGUAGE_FILTER:', LANGUAGE_FILTER)
print('MIN_CONFIDENCE_LANG:', MIN_CONFIDENCE_LANG)
print('Ticker count configured:', len(BANK_TICKERS))

ENTITY_MODE: mask
LANGUAGE_FILTER: en
MIN_CONFIDENCE_LANG: 0.5
Ticker count configured: 29


In [12]:
def is_english(text: str, lang_code: str = 'en', min_confidence: float = 0.5) -> bool:
    """
    Detect if text is in the specified language using langdetect.
    Returns True if the language matches with sufficient confidence.
    """
    if not text or len(text.strip()) < 3:
        return False
    
    try:
        detected_langs = detect_langs(text)
        for lang_prob in detected_langs:
            if lang_prob.lang == lang_code and lang_prob.prob >= min_confidence:
                return True
        return False
    except LangDetectException:
        # If detection fails, assume non-English
        return False

def normalize_text(text: str) -> str:
    text = '' if pd.isna(text) else str(text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+\|\s*(Reuters|Bloomberg|CNBC|Yahoo|MarketWatch|Barrons?)\b.*$', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+-\s*(Reuters|Bloomberg|CNBC|Yahoo|MarketWatch|Barrons?)\b.*$', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## Apply preprocessing

In [13]:
required_cols = ['stock', 'headline', 'date']
missing = [c for c in required_cols if c not in data.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Short-text filtering thresholds
MIN_HEADLINE_CHARS = 20
MIN_HEADLINE_TOKENS = 4

df = data.copy()
df = df.dropna(subset=['stock', 'headline', 'date']).copy()
df['stock'] = df['stock'].astype(str).str.upper().str.strip()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df['date_only'] = df['date'].dt.date

# Preserve raw headline and create variants
df['headline_raw'] = df['headline'].astype(str).str.strip()

# Filter for English headlines BEFORE preprocessing
print('Detecting language of headlines...')
df['is_english'] = df['headline_raw'].apply(
    lambda x: is_english(x, lang_code=LANGUAGE_FILTER, min_confidence=MIN_CONFIDENCE_LANG)
)
rows_before_lang = len(df)
df = df[df['is_english']].copy()
rows_after_lang = len(df)
print(f'Language filter: removed {rows_before_lang - rows_after_lang} non-English headlines')
print(f'Remaining rows: {rows_after_lang}')

df['headline_clean_keep'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='keep'), axis=1)
df['headline_clean_mask'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='mask'), axis=1)
df['headline_clean_remove'] = df.apply(lambda r: preprocess_headline(r['headline_raw'], r['stock'], mode='remove'), axis=1)

if ENTITY_MODE == 'keep':
    df['headline_model'] = df['headline_clean_keep']
elif ENTITY_MODE == 'remove':
    df['headline_model'] = df['headline_clean_remove']
else:
    df['headline_model'] = df['headline_clean_mask']

# Remove empty model headlines
df['headline_model'] = df['headline_model'].fillna('').str.strip()
df = df[df['headline_model'].str.len() > 0].copy()

# Remove very short headlines (chars + token count)
rows_before_short_filter = len(df)
headline_token_count = df['headline_model'].str.split().str.len()
short_mask = (df['headline_model'].str.len() < MIN_HEADLINE_CHARS) | (headline_token_count < MIN_HEADLINE_TOKENS)
removed_short = int(short_mask.sum())
df = df[~short_mask].copy()

# Remove duplicates after filtering
df = df.drop_duplicates(subset=['stock', 'date', 'headline_model']).sort_values('date').reset_index(drop=True)

print(f'Removed short headlines: {removed_short} (min chars={MIN_HEADLINE_CHARS}, min tokens={MIN_HEADLINE_TOKENS})')
print('Processed shape:', df.shape)
display(df[['stock', 'date', 'date_only', 'headline_raw', 'headline_model']].head(10))

Detecting language of headlines...
Language filter: removed 5526 non-English headlines
Remaining rows: 121415
Language filter: removed 5526 non-English headlines
Remaining rows: 121415
Removed short headlines: 226 (min chars=20, min tokens=4)
Processed shape: (121186, 10)
Removed short headlines: 226 (min chars=20, min tokens=4)
Processed shape: (121186, 10)


,stock,date,date_only,headline_raw,headline_model
0,NDAQ,2014-02-17 05:11:00+00:00,2014-02-17,Ameriabank Becomes Market Maker of IFC and EBR...,Ameriabank Becomes Market Maker of IFC and EBR...
1,NDAQ,2014-02-21 05:35:00+00:00,2014-02-21,9th Issue of Corporate Bonds By National Mortg...,9th Issue of Corporate Bonds By National Mortg...
2,NDAQ,2014-04-23 12:39:00+00:00,2014-04-23,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...
3,NDAQ,2014-05-15 10:23:00+00:00,2014-05-15,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...
4,NDAQ,2014-05-26 05:59:00+00:00,2014-05-26,Armswissbank Becomes Market Maker of the 10th ...,Armswissbank Becomes Market Maker of the 10th ...
5,NDAQ,2014-05-26 06:05:00+00:00,2014-05-26,Converse Bank Becomes Account Operator of Fund...,Converse Bank Becomes Account Operator of Fund...
6,NDAQ,2014-05-30 06:27:00+00:00,2014-05-30,ARARATBANK Announced New Issue of Foreign Curr...,ARARATBANK Announced New Issue of Foreign Curr...
7,NDAQ,2014-06-12 09:10:00+00:00,2014-06-12,Inecobank Becomes Account Operator of Funded P...,Inecobank Becomes Account Operator of Funded P...
8,NDAQ,2014-06-26 13:26:00+00:00,2014-06-26,11th Issue of Corporate Bonds By National Mort...,11th Issue of Corporate Bonds By National Mort...
9,NDAQ,2014-06-26 13:33:00+00:00,2014-06-26,ARARATBANK’s USD Denominated Bonds Listed at N...,ARARATBANK’s USD Denominated Bonds Listed at N...


## Quality checks

## Language Detection Results

In [14]:
print(f'\n✅ Language Filter Applied: {LANGUAGE_FILTER} only')
print(f'   Minimum confidence threshold: {MIN_CONFIDENCE_LANG}')
print(f'   Non-English rows removed: {rows_before_lang - rows_after_lang}')
print(f'   Final dataset (English only): {len(df)} rows')
print(f'\nLanguage detection successfully filtered dataset.')
if rows_before_lang > rows_after_lang:
    pct_removed = 100 * (rows_before_lang - rows_after_lang) / rows_before_lang
    print(f'   Percentage removed: {pct_removed:.1f}%')


✅ Language Filter Applied: en only
   Minimum confidence threshold: 0.5
   Non-English rows removed: 5526
   Final dataset (English only): 121186 rows

Language detection successfully filtered dataset.
   Percentage removed: 4.4%


In [15]:
summary = {
    'rows': len(df),
    'unique_stocks': int(df['stock'].nunique()),
    'date_min': df['date'].min(),
    'date_max': df['date'].max(),
    'empty_model_headlines': int((df['headline_model'].str.len() == 0).sum())
}
print(summary)

print('\nTop stocks by article count:')
display(df['stock'].value_counts().head(15).to_frame('n_articles'))

df['raw_len'] = df['headline_raw'].str.len()
df['model_len'] = df['headline_model'].str.len()
print('\nHeadline length stats (raw vs model):')
display(df[['raw_len', 'model_len']].describe().T)

print('\nExamples of transformations:')
display(df[['stock', 'headline_raw', 'headline_clean_mask', 'headline_clean_remove', 'headline_model']].head(10))

{'rows': 121186, 'unique_stocks': 39, 'date_min': Timestamp('2014-02-17 05:11:00+0000', tz='UTC'), 'date_max': Timestamp('2026-04-11 17:07:00+0000', tz='UTC'), 'empty_model_headlines': 0}

Top stocks by article count:


,n_articles
stock,
JPM,12716
GS,9760
BAC,9552
BLK,9280
C,7832
MS,7397
WFC,6456
HSBC,5281
NDAQ,5004



Headline length stats (raw vs model):


,count,mean,std,min,25%,50%,75%,max
raw_len,121186.0,71.124082,24.884420,19.0,57.0,66.0,81.0,504.0
model_len,121186.0,72.472918,25.311563,20.0,58.0,68.0,82.0,504.0



Examples of transformations:


,stock,headline_raw,headline_clean_mask,headline_clean_remove,headline_model
0,NDAQ,Ameriabank Becomes Market Maker of IFC and EBR...,Ameriabank Becomes Market Maker of IFC and EBR...,Ameriabank Becomes Market Maker of IFC and EBR...,Ameriabank Becomes Market Maker of IFC and EBR...
1,NDAQ,9th Issue of Corporate Bonds By National Mortg...,9th Issue of Corporate Bonds By National Mortg...,9th Issue of Corporate Bonds By National Mortg...,9th Issue of Corporate Bonds By National Mortg...
2,NDAQ,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...
3,NDAQ,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...,10th Issue of Corporate Bonds By National Mort...
4,NDAQ,Armswissbank Becomes Market Maker of the 10th ...,Armswissbank Becomes Market Maker of the 10th ...,Armswissbank Becomes Market Maker of the 10th ...,Armswissbank Becomes Market Maker of the 10th ...
5,NDAQ,Converse Bank Becomes Account Operator of Fund...,Converse Bank Becomes Account Operator of Fund...,Converse Bank Becomes Account Operator of Fund...,Converse Bank Becomes Account Operator of Fund...
6,NDAQ,ARARATBANK Announced New Issue of Foreign Curr...,ARARATBANK Announced New Issue of Foreign Curr...,ARARATBANK Announced New Issue of Foreign Curr...,ARARATBANK Announced New Issue of Foreign Curr...
7,NDAQ,Inecobank Becomes Account Operator of Funded P...,Inecobank Becomes Account Operator of Funded P...,Inecobank Becomes Account Operator of Funded P...,Inecobank Becomes Account Operator of Funded P...
8,NDAQ,11th Issue of Corporate Bonds By National Mort...,11th Issue of Corporate Bonds By National Mort...,11th Issue of Corporate Bonds By National Mort...,11th Issue of Corporate Bonds By National Mort...
9,NDAQ,ARARATBANK’s USD Denominated Bonds Listed at N...,ARARATBANK’s USD Denominated Bonds Listed at N...,ARARATBANK’s USD Denominated Bonds Listed at N...,ARARATBANK’s USD Denominated Bonds Listed at N...


In [16]:
# Coverage check: any configured bank tickers still visible in headline_model?
remaining_counts = {}
for ticker in BANK_TICKERS:
    pattern = rf'(?<!_)\b{re.escape(ticker)}\b(?!_)'
    count = int(df['headline_model'].astype(str).str.contains(pattern, regex=True).sum())
    if count > 0:
        remaining_counts[ticker] = count

print('\nRemaining visible configured tickers in headline_model:', len(remaining_counts))
if remaining_counts:
    display(pd.DataFrame(sorted(remaining_counts.items(), key=lambda x: x[1], reverse=True), columns=['ticker', 'count']).head(20))
else:
    print('✅ All configured bank tickers are masked/removed from headline_model.')


Remaining visible configured tickers in headline_model: 0
✅ All configured bank tickers are masked/removed from headline_model.


## Save outputs for downstream pipelines

In [17]:
output_main = 'Data/GSIB_preprocessed.csv'
output_model = 'Data/GSIB_model_input.csv'

df.to_csv(output_main, index=False)

model_df = df[['stock', 'date', 'date_only', 'headline_model', 'headline_raw']].copy()
model_df = model_df.rename(columns={'headline_model': 'headline'})
model_df.to_csv(output_model, index=False)

print('Saved:')
print(f'  {output_main} -> {df.shape}')
print(f'  {output_model} -> {model_df.shape}')

Saved:
  Data/GSIB_preprocessed.csv -> (121186, 12)
  Data/GSIB_model_input.csv -> (121186, 5)
